# 03 — RAG (Retrieval-Augmented Generation) Sistemi

**Kapsam:** Vektör veri tabanları, embedding modelleri ve belge yönetimi bileşenlerini
bir araya getirerek RAG mimarisi tasarlama, geliştirme ve optimize etme.

Bu notebook'ta:
1. Chunk'ları embedding'e çevirip Chroma'ya indeksliyoruz
2. Retrieval + reranking'i deniyoruz
3. Uçtan uca RAG pipeline'ını (retrieve → prompt → generate) çalıştırıyoruz
4. Diyalog yöneticisini (02. notebook) artık gerçek RAG ile test ediyoruz

In [ ]:
# Bu hücre HER notebook'ta ayrı ayrı çalıştırılmalı: Colab'da her sekme/notebook
# genellikle kendi çalışma zamanını (VM) alır, yani /content her seferinde sıfırdanmış
# gibi başlar. Bu hücre kendi kendini onaran bir kurulum yapar:
#   1) Proje klasörü zaten varsa (aynı çalışma zamanında önceki hücre/notebook
#      tarafından kurulmuşsa) unpack adımını atlar.
#   2) Yoksa Google Drive'ı mount edip, DRIVE_ZIP_PATH'teki zip'i /content'e açar
#      (zip'in içinde 'baykar-nlp-hazirlik/' klasörü kök olarak yer almalı).
#   3) Drive'da zip de yoksa, kendi GitHub reponuzu klonlamanız için bir uyarı basar.
#   4) EN ÖNEMLİSİ: data/, models/, mlruns/ klasörlerini Drive'daki kalıcı bir
#      klasöre sembolik bağlantı (symlink) yapar. Neden gerekli: /content her
#      runtime'da sıfırlanır, yani 01. notebook'ta ürettiğiniz corpus.jsonl gibi
#      dosyalar farklı bir runtime'da (örn. 02. notebook'u açtığınızda) KAYBOLUR.
#      Bu adım olmadan her notebook'u ayrı ayrı çalıştırdığınızda önceki adımların
#      ürettiği veriyi bulamazsınız. Sembolik bağlantı sayesinde hangi runtime'da
#      olursanız olun aynı kalıcı depoyu okur/yazarsınız.
import os, sys
os.environ.setdefault("USE_TF", "0")  # transformers TensorFlow'u hic denemesin (Colab'da protobuf catismasi yasatiyor)

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
DRIVE_ZIP_PATH = "/content/drive/MyDrive/baykar-nlp-hazirlik.zip"
DRIVE_DATA_DIR = "/content/drive/MyDrive/baykar-nlp-hazirlik-data"
PERSIST_DIRS = ["data/raw", "data/processed", "models", "mlruns"]  # chroma_db BILEREK haric (asagida)

try:
    from google.colab import drive
    # force_remount=True KULLANMIYORUZ: bu, zaten mount edilmişken bile her seferinde
    # yeniden yetkilendirme (izin penceresi) ister, gereksiz bekleme/kesinti yaratır.
    # drive.mount() zaten mount edilmişse kendi içinde anında geri döner; mount
    # edilmemişse (bu runtime'da ilk çalıştırma) normal şekilde izin ister — bu
    # durumda çıkan izin penceresini/bağlantısını tamamlamanız gerekir, hücreyi
    # durdurmayın.
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False  # Colab dışında (yerelde) çalışıyorsanız Drive adımları atlanır.

if not os.path.exists(PROJECT_DIR) and IN_COLAB:
    if os.path.exists(DRIVE_ZIP_PATH):
        import shutil
        shutil.unpack_archive(DRIVE_ZIP_PATH, "/content")
    else:
        print(f"UYARI: {DRIVE_ZIP_PATH} bulunamadı. Zip'i Drive'ınızın köküne "
              "yükleyin ya da kendi reponuzu klonlayın: "
              f"!git clone <repo-url> {PROJECT_DIR}")

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

if IN_COLAB and os.path.exists(PROJECT_DIR):
    import shutil
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for _name in PERSIST_DIRS:
        _drive_path = os.path.join(DRIVE_DATA_DIR, _name)
        os.makedirs(_drive_path, exist_ok=True)
        _local_path = os.path.join(PROJECT_DIR, _name)
        os.makedirs(os.path.dirname(_local_path), exist_ok=True)  # orn. data/ klasorunu gercek dizin olarak olustur

        if os.path.islink(_local_path):
            continue  # zaten Drive'a bağlanmış

        if os.path.isdir(_local_path):
            # Zip'ten gelen boş klasörü kaldırıp yerine symlink koyuyoruz. İçinde
            # (nadiren) veri varsa önce Drive'a taşıyoruz, hiçbir şeyi kaybetmiyoruz.
            for _item in os.listdir(_local_path):
                _src = os.path.join(_local_path, _item)
                _dst = os.path.join(_drive_path, _item)
                if not os.path.exists(_dst):
                    shutil.move(_src, _dst)
            shutil.rmtree(_local_path)

        os.symlink(_drive_path, _local_path)

    print("Kalıcı veri klasörü:", DRIVE_DATA_DIR)
    print("Not: chroma_db (vektor veritabani) Drive'a BAGLANMADI -- SQLite, Drive'in")
    print("     FUSE dosya sisteminde yazma kilidini desteklemiyor ('OperationalError:")
    print("     attempt to write a readonly database'). Her yeni runtime'da RAG")
    print("     notebook'undaki (03) indeksleme hucresini tekrar calistirin -- chunks.jsonl")
    print("     zaten Drive'da oldugu icin bu hizli ve ucretsiz bir islemdir.")


## Colab ortam düzeltmeleri

Kurulum hücresinden hemen sonra çalıştırın. Chroma, Transformers v5 yamaları ve paket sürümleri.

In [ ]:
import os, subprocess, sys

PROJECT_DIR = "/content/baykar-nlp-hazirlik"
bootstrap = os.path.join(PROJECT_DIR, "scripts", "colab_bootstrap.py")

if not os.path.exists(bootstrap):
    raise FileNotFoundError(
        "scripts/colab_bootstrap.py bulunamadi. "
        "Guncel projeyi zip'leyip Drive'a yukleyin."
    )

subprocess.run([sys.executable, bootstrap], check=True, cwd=PROJECT_DIR)

## 1. İndeksleme

In [ ]:
import json
from src.rag.vector_store import index_chunks

chunks = []
with open("data/processed/chunks.jsonl", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

n = index_chunks(chunks)
print(f"{n} chunk vektör veritabanına indekslendi.")


## 2. Retrieval + Reranking

In [ ]:
from src.rag.retriever import retrieve

results = retrieve("İyi bir API dokümantasyonu hangi bölümleri içermeli?", final_k=5)
for r in results:
    print(f"[{r.get('rerank_score', r['similarity']):.3f}] {r['doc_title']} — {r['text'][:100]}...")


## 3. Uçtan uca RAG

In [ ]:
from src.rag.rag_pipeline import answer

result = answer("Bir kurulum kılavuzu hangi adımları içermeli?")
print("CEVAP:\n", result["answer"])
print("\nKAYNAKLAR:")
for s in result["sources"]:
    print("-", s["title"], s["url"])


## 4. Diyalog üzerinden RAG

In [ ]:
from src.nlp_tasks.dialogue import chat

r1 = chat("demo-session", "İyi bir README nasıl yapılandırılır?")
print(r1["answer"])

r2 = chat("demo-session", "Peki lisans bölümüne ne yazmalıyım?")
print(r2["answer"])
